# Concord Productivity & Wellness Analytics Report

This Jupyter notebook provides a comprehensive data analysis report for the Concord task manager. It connects directly to the Supabase PostgreSQL database to fetch the latest live data, computes productivity metrics, and generates visual analytics charts. If PostgreSQL is not configured, it automatically falls back to the local SQLite database (`concord.db`).

In [ ]:
import sqlite3
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Load environment variables from .env file
load_dotenv()
database_url = os.getenv("DATABASE_URL")
conn = None

if database_url:
    print("DATABASE_URL found. Connecting directly to PostgreSQL...")
    try:
        if "localhost" in database_url:
            database_url = database_url.replace("localhost", "127.0.0.1")
        conn = psycopg2.connect(database_url)
        print("Successfully connected to PostgreSQL database!")
    except Exception as e:
        print(f"PostgreSQL connection failed: {e}")
        conn = None

if conn is None:
    db_path = "concord.db"
    if not os.path.exists(db_path):
        print(f"Database '{db_path}' not found and no DATABASE_URL defined in .env.")
    else:
        conn = sqlite3.connect(db_path)
        print("Successfully connected to SQLite fallback database!")

## 1. Load Data Tables

In [ ]:
df_users = pd.read_sql_query("SELECT * FROM users", conn)
df_profiles = pd.read_sql_query("SELECT * FROM user_profiles", conn)
df_tasks = pd.read_sql_query("SELECT * FROM tasks", conn)
df_mood = pd.read_sql_query("SELECT * FROM mood_history", conn)

# Query chat history/messages supporting both table naming schemes
df_chat = pd.DataFrame()
for table_name in ["chat_messages", "chat_history"]:
    try:
        df_chat = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        break
    except Exception:
        continue

print(f"Users logged: {len(df_users)}")
print(f"Profiles logged: {len(df_profiles)}")
print(f"Tasks logged: {len(df_tasks)}")
print(f"Mood logs: {len(df_mood)}")
print(f"Chat message count: {len(df_chat)}")

## 2. General Data Summaries & Missing Values

In [ ]:
print("--- Tasks Info ---")
print(df_tasks.info())
print("\n--- Missing Values ---")
print(df_tasks.isnull().sum())

## 3. Productivity Metrics Calculations

In [ ]:
# Preprocess date fields
df_tasks['created_at_dt'] = pd.to_datetime(df_tasks['created_at'], format='mixed', errors='coerce', utc=True)
df_tasks['updated_at_dt'] = pd.to_datetime(df_tasks['updated_at'], format='mixed', errors='coerce', utc=True)

total_tasks = len(df_tasks)
completed_tasks = len(df_tasks[df_tasks['status'] == 'completed'])
completion_rate = (completed_tasks / total_tasks * 100) if total_tasks > 0 else 0

print(f"Overall Task Completion Rate: {completion_rate:.2f}%")

# Priority-wise breakdown
prio_stats = df_tasks.groupby('priority').apply(
    lambda x: pd.Series({
        'Total': len(x),
        'Completed': (x['status'] == 'completed').sum(),
        'Completion Rate (%)': round((x['status'] == 'completed').mean() * 100, 2)
    })
)
print("\nPriority Productivity Stats:")
print(prio_stats)

## 4. Visualizations

In [ ]:
# Task Status Distribution
plt.figure(figsize=(6, 5))
status_counts = df_tasks['status'].value_counts()
plt.pie(status_counts, labels=status_counts.index.str.replace('_', ' ').str.capitalize(), autopct='%1.1f%%', colors=['#10b981', '#3b82f6', '#f59e0b'][:len(status_counts)], startangle=140, wedgeprops={'edgecolor': 'white'})
plt.title('Task Status Distribution')
plt.show()

# Priority Distribution
plt.figure(figsize=(7, 5))
priority_counts = df_tasks['priority'].value_counts().reindex(['low', 'medium', 'high']).fillna(0)
sns.barplot(x=priority_counts.index, y=priority_counts.values, palette=['#10b981', '#f59e0b', '#ef4444'])
plt.title('Priority Distribution')
plt.ylabel('Number of Tasks')
plt.show()

In [ ]:
# Tasks and completion rate by category
plt.figure(figsize=(8, 5))
cat_counts = df_tasks['category'].value_counts()
sns.barplot(y=cat_counts.index, x=cat_counts.values, palette="crest")
plt.title('Tasks by Category')
plt.xlabel('Number of Tasks')
plt.show()

## 5. Mood and Wellness Trends

In [ ]:
if not df_mood.empty:
    # Mood log frequencies
    plt.figure(figsize=(7, 5))
    mood_counts = df_mood['label'].value_counts()
    sns.barplot(x=mood_counts.index, y=mood_counts.values, palette="pastel")
    plt.title('Mood Frequency Distribution')
    plt.ylabel('Logs')
    plt.show()

## 6. Cycle-Phase Analysis for Female Users

In [ ]:
female_profiles = df_profiles[df_profiles['gender'] == 'female']
if not female_profiles.empty and not df_tasks.empty and female_profiles.iloc[0]['last_period_date']:
    cycle_len = female_profiles.iloc[0]['cycle_length'] or 28
    luteal_len = female_profiles.iloc[0]['luteal_phase_length'] or 14
    last_period = pd.to_datetime(female_profiles.iloc[0]['last_period_date'], format='mixed', errors='coerce', utc=True)
    
    phases = []
    for idx, row in df_tasks.iterrows():
        days_since = (row['created_at_dt'] - last_period).days
        day_in_cycle = (days_since % cycle_len) + 1
        
        menstrual_end = 5
        follicular_end = cycle_len - luteal_len - 1
        ovulatory_day = cycle_len - luteal_len
        
        phase = "Luteal"
        if day_in_cycle <= menstrual_end:
            phase = "Menstrual"
        elif day_in_cycle <= follicular_end:
            phase = "Follicular"
        elif day_in_cycle == ovulatory_day:
            phase = "Ovulatory"
        phases.append(phase)
        
    df_tasks['cycle_phase'] = phases
    phase_comp = df_tasks.groupby('cycle_phase').apply(lambda x: (x['status'] == 'completed').mean() * 100).reindex(['Menstrual', 'Follicular', 'Ovulatory', 'Luteal']).fillna(0)
    
    plt.figure(figsize=(7, 5))
    sns.barplot(x=phase_comp.index, y=phase_comp.values, palette="rocket")
    plt.title('Completion Rate by Menstrual Cycle Phase (%)')
    plt.ylabel('Completion Rate (%)')
    plt.ylim(0, 100)
    plt.show()

## 7. Close Connection

In [ ]:
conn.close()
print("Database connection closed. Analysis complete!")